# 01 - 手写 MiniLlama

> 配套笔记：[../notes/01_Transformer 详解.md](../notes/01_Transformer%20详解.md) ｜ 知乎：<https://zhuanlan.zhihu.com/p/28364382951>

LLaMA 简化版（Decoder-Only），五块拼出来：

1. **MHA** — 带 RoPE
2. **MLP** — SwiGLU
3. **DecoderBlock** — Pre-Norm + 残差
4. **MiniLlama** — Embedding → N×Block → RMSNorm → Projection
5. **Smoke test**

## 0. 环境

直接复用：
- `x_transformers.RMSNorm`
- `rotary_embedding_torch.RotaryEmbedding`（RoPE）

In [ ]:
!pip install x-transformers
!pip install rotary-embedding-torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from x_transformers import RMSNorm
from rotary_embedding_torch import RotaryEmbedding

## 1. MHA（带 RoPE）

LLaMA 版与标准 MHA 的区别：
- **位置编码**：RoPE 直接旋转 Q/K，不在 embedding 上加
- **Linear 不带 bias**（后接 RMSNorm，bias 被消掉）

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads

        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

        self.rope = RotaryEmbedding(dim=self.head_dim)

    def forward(self, x, mask=None):
        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # 拆头：[B, T, H, d] -> [B, H, T, d]
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # RoPE 旋转 Q/K
        q = self.rope(q)[:, :, :, :, 0]
        k = self.rope(k)[:, :, :, :, 0]

        attn_scores = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)

        if mask is not None:
            mask = mask.unsqueeze(1)  # [B,T,T] -> [B,1,T,T]
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_output = attn_weights @ v

        # 拼头：[B, H, T, d] -> [B, T, hidden]
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, -1)

        return self.out_proj(attn_output)

## 2. MLP（SwiGLU）

LLaMA 用 SwiGLU 替掉 ReLU：

$$\mathrm{SwiGLU}(x) = \mathrm{SiLU}(W_g x) \odot (W_u x)$$

三个 Linear：
- `gate_proj` → SiLU 当门控
- `up_proj` → 内容
- 按位相乘后 `out_proj` 降维

In [ ]:
# 标准版 FFN（参考用）
"""
class SimpleMLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or 4 * dim
        self.fc1 = nn.Linear(dim, hidden_dim, bias=False)
        self.fc2 = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))
"""


class MLP(nn.Module):
    def __init__(self, dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or 4 * dim
        self.gate_proj = nn.Linear(dim, hidden_dim, bias=False)
        self.up_proj   = nn.Linear(dim, hidden_dim, bias=False)
        self.out_proj  = nn.Linear(hidden_dim, dim, bias=False)

    def forward(self, x):
        return self.out_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

## 3. DecoderBlock（Pre-Norm + 残差）

LLaMA 改 Post-Norm 为 Pre-Norm：
- 标准：`x → Sublayer → +x → Norm`
- LLaMA：`x → Norm → Sublayer → +x`

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, dim=512, num_heads=8):
        super().__init__()
        self.attn = MultiHeadAttention(dim, num_heads)
        self.ffn  = MLP(dim)
        self.norm1 = RMSNorm(dim)
        self.norm2 = RMSNorm(dim)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## 4. MiniLlama 整体

`Embedding → N × DecoderBlock → RMSNorm → Linear`

注意最后是 **先 RMSNorm 再投影**（与 GPT/BERT 的 `Linear → LayerNorm` 顺序相反）。

In [ ]:
class MiniLlama(nn.Module):
    def __init__(self, vocab_size=32000, dim=512, depth=8, num_heads=8):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, dim)
        self.blocks = nn.ModuleList([DecoderBlock(dim, num_heads) for _ in range(depth)])
        self.norm   = RMSNorm(dim)
        self.final_proj = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, x, mask=None):
        x = self.embed(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.norm(x)
        return self.final_proj(x)

## 5. Smoke Test

预期：输入 `[2, 16]` → 输出 `[2, 16, 32000]`。

In [ ]:
def test_mini_llama():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MiniLlama(vocab_size=32000, dim=512, depth=8, num_heads=8).to(device)

    batch_size, seq_len = 2, 16
    dummy_input = torch.randint(0, 32000, (batch_size, seq_len)).to(device)
    dummy_mask  = torch.ones((batch_size, seq_len, seq_len), dtype=torch.float32).to(device)

    output = model(dummy_input, mask=dummy_mask)

    print("输入形状:", dummy_input.shape)
    print("输出形状:", output.shape)

test_mini_llama()